# Exploration: can this MMM be trusted, and for which channels?

An MMM estimates something that is never observed, so on real data its output cannot be
scored. This project simulates from a process whose parameters are written down, which turns
"does it look reasonable" into "is it right".

**The data here is simulated on purpose.** No real company's spend or revenue appears
anywhere in this repository.

In [1]:
import sys
sys.path.insert(0, "src")

import pandas as pd

from mmm.config import BASELINE, TRUTH
from mmm.model import signal_to_noise
from mmm.simulate import simulate, true_roi

frame = simulate()
print(f"{len(frame)} weeks, {len(TRUTH)} channels")
print(f"weekly revenue: {frame['revenue'].mean():,.0f} on average")

104 weeks, 4 channels
weekly revenue: 86,679 on average


/tmp/claude-1000/-home-ariel-Documentos-secondBrain-ronco-app/fc6efcad-21d0-483b-8f07-61bce5d66383/scratchpad/projects/marketing-mix-modeling/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. The truth the model is not told

In [2]:
truth = pd.DataFrame([
    {"channel": name, "adstock_alpha": t.adstock_alpha,
     "saturation_lambda": t.saturation_lambda, "beta": t.beta}
    for name, t in TRUTH.items()
])
print(truth.to_string(index=False))
print()
print(true_roi(frame)[["channel", "spend", "contribution", "roi"]].round(4).to_string(index=False))

  channel  adstock_alpha  saturation_lambda  beta
       tv           0.70                2.5  0.55
   search           0.15                4.0  0.40
   social           0.35                3.0  0.28
affiliate           0.05                6.0  0.09

  channel        spend  contribution    roi
       tv 2835066.3528  2033092.1156 0.7171
   search 1898075.8923  1307140.4611 0.6887
   social 1445305.8456   561608.7460 0.3886
affiliate  640266.9154   110532.2521 0.1726


## 2. Media must not explain everything

The baseline carries trend, yearly seasonality and noise. Without them the test would be a
softball: a model that credits a channel with the season it spent in would still look
accurate.

In [3]:
media = sum(frame[f"{c}_contribution_true"].sum() for c in TRUTH)
print(f"media share of revenue:    {media / frame['revenue'].sum():.1%}")
print(f"baseline the model must NOT claim: {1 - media / frame['revenue'].sum():.1%}")

media share of revenue:    44.5%
baseline the model must NOT claim: 55.5%


## 3. The question to ask before fitting anything

Not every channel is measurable. A channel whose weekly contribution sits below the weekly
revenue noise carries no information about its own ROI, and the model will return the prior
with a confident-looking interval attached.

In [4]:
diagnostic = signal_to_noise(frame, BASELINE.noise_sd)
diagnostic["share_of_revenue"] = diagnostic["share_of_revenue"].map(lambda v: f"{v:.1%}")
print(diagnostic.round(4).to_string(index=False))
print()
print("affiliate's weekly contribution varies by about 4% of the revenue noise.")
print("There is nothing in this data for a model to learn about its ROI.")

  channel share_of_revenue  signal_to_noise
       tv            22.6%           0.8810
   search            14.5%           0.4540
   social             6.2%           0.2961
affiliate             1.2%           0.0385

affiliate's weekly contribution varies by about 4% of the revenue noise.
There is nothing in this data for a model to learn about its ROI.


## 4. What the fitted model recovered

In [5]:
recovery = pd.read_csv("reports/roi_recovery.csv")
view = recovery[["channel", "true_roi", "estimated_roi", "roi_error_pct",
                 "true_rank", "estimated_rank"]].round(4)
print(view.to_string(index=False))

from scipy import stats
all_channels = stats.spearmanr(recovery["true_roi"], recovery["estimated_roi"]).statistic
identifiable = recovery[recovery["channel"] != "affiliate"]
measurable = stats.spearmanr(identifiable["true_roi"], identifiable["estimated_roi"]).statistic

print()
print(f"ROI rank correlation, all channels:        {all_channels:+.3f}")
print(f"ROI rank correlation, excluding affiliate: {measurable:+.3f}")
print(f"mean absolute ROI error, excluding it:      {identifiable['roi_error'].abs().mean():.4f}")

  channel  true_roi  estimated_roi  roi_error_pct  true_rank  estimated_rank
       tv    0.7171         1.0123         0.4116          1               2
   search    0.6887         0.8122         0.1794          2               3
   social    0.3886         0.3237        -0.1671          3               4
affiliate    0.1726         1.2714         6.3648          4               1

ROI rank correlation, all channels:        -0.200
ROI rank correlation, excluding affiliate: +1.000
mean absolute ROI error, excluding it:      0.1612


The model gets **every measurable channel in the right order**. The overall ranking inverts
because of the single channel the diagnostic already flagged as unmeasurable.

## 5. Why that one bad estimate is not harmless

In [6]:
allocation = pd.read_csv("reports/budget_allocation.csv")
merged = allocation.merge(
    true_roi(frame)[["channel", "roi"]].rename(columns={"roi": "true_roi"}), on="channel"
)
merged["true_rank"] = merged["true_roi"].rank(ascending=False).astype(int)
print(merged[["channel", "true_rank", "current_weekly_spend",
              "recommended_weekly_spend", "change_pct", "at_bound"]].to_string(index=False))

  channel  true_rank  current_weekly_spend  recommended_weekly_spend  change_pct  at_bound
       tv          1              26174.26                  17892.92     -0.3164     False
   search          2              20974.33                  17892.92     -0.1469     False
   social          3              12120.64                  17892.92      0.4762     False
affiliate          4               5590.45                  11180.90      1.0000      True


**The optimiser doubles spend on the worst channel**, stopped only by the 200% bound, while
cutting the second-best by 42%.

The machinery is working: the three measurable channels converge toward equal spend, which is
the optimiser correctly equalising marginal returns. It is simply being fed one number it
should never have received.

**The practical rule:** compute signal-to-noise per channel first. Anything below roughly 0.1
should be pinned to its current spend and excluded from the optimisation, not debated in a
meeting. Measuring a small channel needs a geo experiment, not a bigger model.